In [1]:
import paths, dataframes

In [2]:
from pipe import select
import pandas

In [3]:
every_hive_audio_features = pandas.concat(
    paths.all_audio_features_csv_filepaths(hive_numbers=[1,2,3,4,5,6])
    | select(dataframes.from_filepath),
    ignore_index=True,
)

In [4]:
every_hive_audio_features.drop(columns=[c for c in every_hive_audio_features.columns if c.startswith("Unnamed")], inplace=True)
every_hive_audio_features.head()

,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flatness_mean,...,low_to_high_energy_ratio_mean,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency
0,2026-03-07 17:00:00,hive_01,17-18,1436.880370,254.820748,2126.972494,213.651310,3888.668335,983.172022,0.009998,...,3.964906,1.821179,2.771714,0.447971,3.347682e+06,2.031807,3.600953e+06,1.288793,1.945159e+06,1.108633
1,2026-03-07 18:00:00,hive_01,18-19,1556.936054,206.016975,2218.712329,140.366414,4486.925926,606.595246,0.012219,...,4.070238,1.513359,2.624779,0.288759,4.158193e+04,1.123286,4.094515e+04,7.488570,2.257747e+04,3.561997
2,2026-03-08 07:00:00,hive_01,07-08,1339.847368,195.620321,1994.812622,145.244711,3470.113293,668.707598,0.008959,...,5.196596,2.051203,2.904476,0.475835,6.614851e+05,1.731035,5.762245e+05,1.229742,2.914703e+05,5.090786
3,2026-03-08 08:00:00,hive_01,08-09,1577.337881,246.118354,2120.288059,278.308385,4233.065704,638.337021,0.034418,...,3.357718,0.948413,2.254672,0.352719,6.197887e+06,1.076116,1.233090e+07,1.525840,1.318154e+07,1.588896
4,2026-03-08 09:00:00,hive_01,09-10,1364.667436,201.022314,2051.912945,157.315150,3685.908598,722.311976,0.008240,...,5.303287,1.565435,2.646992,0.361065,2.875560e+06,1.369173,2.206735e+06,3.983351,1.164676e+06,1.424729


In [5]:
every_hive_audio_features["queenlessness"] = every_hive_audio_features.apply(dataframes.queenstate_from_row, axis=1) == "queenless"

In [6]:
ambient_features = pandas.read_csv(
    paths.ambient_features_filepath, parse_dates=["timestamp"]
)

In [7]:
ambient_features.drop(columns=[c for c in ambient_features.columns if c.startswith("Unnamed")], inplace=True)

In [8]:
merged_features_dataframe = every_hive_audio_features.merge(ambient_features, on="timestamp", how="left")

In [9]:
dataframes.saved_csv_filepath_from_features_dataframe(
    dataframe=merged_features_dataframe,
    dataframe_filename=paths.all_merged_features_filename,
)

'../../data/features/all_merged_features.csv'

In [10]:
import os
assert os.path.isfile(paths.all_merged_features_filepath)

In [11]:
# which columns have NaN, and how many
merged_features_dataframe.isnull().sum()[merged_features_dataframe.isnull().sum() > 0]

Series([], dtype: int64)

In [12]:
hive_04_audio = every_hive_audio_features[every_hive_audio_features["hive"] == "hive_04"]
hive_04_audio[hive_04_audio["timestamp"].duplicated(keep=False)].sort_values("timestamp")

,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flatness_mean,...,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency,queenlessness
363,2026-03-11 11:00:00,hive_04,11-12,1368.078646,1144.728810,1504.142659,1244.465828,3530.582075,2933.536394,0.446580,...,0.789346,0.715014,0.604416,2.842446e+06,1.030410,6.480053e+06,1.036192,5.707569e+06,1.451183,True
365,2026-03-11 11:00:00,hive_04,11-12,779.978677,1185.208613,818.262858,1237.338400,1961.178762,2967.471857,0.720049,...,0.474850,0.378094,0.582952,4.155778e+05,1.002254,1.300785e+06,1.129987,1.101111e+06,3.330017,True
367,2026-03-11 11:00:00,hive_04,11-12,2125.901775,321.684469,2490.561931,185.633981,5661.307428,721.977536,0.040812,...,0.980476,1.259226,0.172823,3.813410e+05,1.028347,9.082342e+05,7.227617,7.660711e+05,1.100452,True
364,2026-03-11 12:00:00,hive_04,12-13,864.127659,1120.211718,962.517506,1239.953191,2260.209630,2918.616184,0.646637,...,0.758242,0.455457,0.591797,2.116199e+06,19.387571,4.287508e+06,1.335988,3.721193e+06,1.053015,True
366,2026-03-11 12:00:00,hive_04,12-13,433.722524,905.928603,480.698734,999.106616,1133.271147,2359.981274,0.824750,...,0.382650,0.222373,0.466070,2.774778e+06,1.315317,7.879411e+06,1.278335,6.616982e+06,1.042481,True
368,2026-03-11 12:00:00,hive_04,12-13,2223.819935,303.971177,2522.479257,153.863705,5865.268433,612.846074,0.052644,...,0.645405,1.242613,0.238823,1.317076e+06,1.158944,3.177824e+06,1.429172,2.774196e+06,2.085031,True


In [13]:
hive_04_csv = paths.features_folderpath + paths.audio_features_filename_from_hive_number(4)
pandas.read_csv(hive_04_csv).query("timestamp == '2026-03-11 11:00:00'")

,timestamp,hive,time_slice,spectral_centroid_mean,spectral_centroid_std,spectral_bandwidth_mean,spectral_bandwidth_std,spectral_rolloff_mean,spectral_rolloff_std,spectral_flatness_mean,...,low_to_high_energy_ratio_mean,low_to_high_energy_ratio_std,middle_to_high_energy_ratio_mean,middle_to_high_energy_ratio_std,low_modulation_energy,low_peak_modulation_frequency,middle_modulation_energy,middle_peak_modulation_frequency,high_modulation_energy,high_peak_modulation_frequency
33,2026-03-11 11:00:00,hive_04,11-12,1368.078646,1144.728810,1504.142659,1244.465828,3530.582075,2933.536394,0.446580,...,0.553439,0.789346,0.715014,0.604416,2.842446e+06,1.030410,6.480053e+06,1.036192,5.707569e+06,1.451183
35,2026-03-11 11:00:00,hive_04,11-12,779.978677,1185.208613,818.262858,1237.338400,1961.178762,2967.471857,0.720049,...,0.252219,0.474850,0.378094,0.582952,4.155778e+05,1.002254,1.300785e+06,1.129987,1.101111e+06,3.330017
37,2026-03-11 11:00:00,hive_04,11-12,2125.901775,321.684469,2490.561931,185.633981,5661.307428,721.977536,0.040812,...,0.944570,0.980476,1.259226,0.172823,3.813410e+05,1.028347,9.082342e+05,7.227617,7.660711e+05,1.100452
